In [5]:
!pip install -q -U torch transformers peft datasets bitsandbytes trl accelerate
!pip install -q git+https://github.com/yuchenlin/LLM-Blender.git # PairRM
!pip install -q pandas

  Preparing metadata (setup.py) ... done


In [4]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: write).
The token `Colab Notebook` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-cred

Phase 1: Generating Raw Pairs (with Llama-3.2)

In [6]:
import torch
from transformers import AutoTokenizer, pipeline
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

#Configuration
model_id="meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading model: {model_id}...")

tokenizer=AutoTokenizer.from_pretrained(model_id)
generator=pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

# 2. get prompts (subset of 100 for testing) - we use Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train[:50]")
print(f"Loaded {len(dataset)} prompts.")

# 3. Generate Response Pairs
raw_data=[]
print("Generating pairs...")

for item in tqdm(dataset):
    user_content=item["instruction"]
    if item['input']:
      user_content+=f"\nContent: {item['input']}" # Changed 'inputs' to 'input'

    #format as chant messages so model understands its an instruction
    messages=[{"role":"user","content":user_content}]
    prompt_formatted=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)

    #generate 2 response
    output=generator(prompt_formatted,
                     max_new_tokens=256, # Changed max_tokens to max_new_tokens
                     do_sample=True,
                     temperature=1.1,
                     top_p=0.9,
                     eos_token_id=tokenizer.eos_token_id,
                     pad_token_id=tokenizer.eos_token_id,
                     return_full_text=False, #get only aswer not prompt
                     num_return_sequences=2 # Generate two sequences for comparison
                    )
    response_a=output[0]['generated_text'].strip()
    response_b=output[1]['generated_text'].strip()

    raw_data.append({
        "prompt": user_content, # Save the raw prompt for the training file later
        "response_a": response_a,
        "response_b": response_b
    })

    #save tocsv
    df_raw=pd.DataFrame(raw_data)
    df_raw.to_csv("raw_pairs.csv",index=False)
    print("\nsuccess! raw_pairs.csv saved")
    print(f"\nexample pair: ")
    print(f"prompt: {df_raw.iloc[0]['prompt'][:50]}")
    print(f"response A:{df_raw.iloc[0]['response_a'][:50]}")
    print(f"respose B: {df_raw.iloc[0]['response_b'][:50]}")

Loading model: meta-llama/Llama-3.2-1B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...


success! raw_pairs.csv saved

example pair: 
prompt: Give three tips for staying healthy.
response A:Here are three tips for staying healthy:

1. **Hyd
respose B: Here are three tips for maintaining overall health


 98%|█████████▊| 49/50 [04:44<00:04,  4.70s/it]


success! raw_pairs.csv saved

example pair: 
prompt: Give three tips for staying healthy.
response A:Here are three tips for staying healthy:

1. **Hyd
respose B: Here are three tips for maintaining overall health


100%|██████████| 50/50 [04:46<00:00,  5.72s/it]


success! raw_pairs.csv saved

example pair: 
prompt: Give three tips for staying healthy.
response A:Here are three tips for staying healthy:

1. **Hyd
respose B: Here are three tips for maintaining overall health


Phase 2: Judge

In [7]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.4 MB/s eta 0:00:00


In [16]:
import pandas as pd
from groq import Groq
import time
from tqdm import tqdm

# --- CONFIGURATION ---
# Use the Key you provided
GROQ_API_KEY = "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"
# ---------------------

# 1. Load the Raw Data
try:
    df = pd.read_csv("raw_pairs.csv")
    print(f"Loaded {len(df)} pairs to judge.")
except FileNotFoundError:
    print("Error: 'raw_pairs.csv' not found. Please run Phase 1 first.")

# 2. Setup the Judge Client
client = Groq(api_key=GROQ_API_KEY)

# 3. Define the Judge Prompt
JUDGE_SYSTEM_PROMPT = """
You are an impartial, expert AI evaluator.
Your job is to compare two responses (Response A and Response B) to a given User Instruction.
You must select the response that is more helpful, accurate, and safe.

Output Format:
1. Analysis: Briefly explain why one response is better.
2. Winner: Output strictly 'A' or 'B'.
"""

def judge_pair(prompt, res_a, res_b):
    user_message = f"""
    Instruction: {prompt}

    [Response A]:
    {res_a}

    [Response B]:
    {res_b}
    """

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            # We use the 70B model because the Judge must be smarter than the student
            model="llama-3.3-70b-versatile",
            temperature=0.0,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return "Error"

# 4. Run the Judging Process
preference_data = []
print("Judging pairs... (This is fast with Groq)")

for index, row in tqdm(df.iterrows(), total=len(df)):
    judgment = judge_pair(row['prompt'], row['response_a'], row['response_b'])

    # Parse the winner
    chosen = None
    rejected = None

    # Check for Winner A or B
    if "Winner: A" in judgment or "Winner: 'A'" in judgment:
        chosen = row['response_a']
        rejected = row['response_b']
    elif "Winner: B" in judgment or "Winner: 'B'" in judgment:
        chosen = row['response_b']
        rejected = row['response_a']

    # Only save if a clear winner was found
    if chosen:
        preference_data.append({
            "prompt": row['prompt'],
            "chosen": chosen,
            "rejected": rejected,
            "judgment_reasoning": judgment
        })

    # Sleep briefly to be nice to the API
    time.sleep(0.5)

# 5. Save the Final Dataset
df_pref = pd.DataFrame(preference_data)
df_pref.to_json("preference_dataset.jsonl", orient="records", lines=True)

print(f"\nSuccess! Created {len(df_pref)} preference pairs.")
print("Saved to 'preference_dataset.jsonl'")

if not df_pref.empty:
    print("\nExample Judgment:")
    print(df_pref.iloc[0]['judgment_reasoning'])
else:
    print("\nWarning: No pairs were created. Check the parsing logic or API connection.")

Loaded 50 pairs to judge.
Judging pairs... (This is fast with Groq)


100%|██████████| 50/50 [01:35<00:00,  1.91s/it]


Success! Created 50 preference pairs.
Saved to 'preference_dataset.jsonl'

Example Judgment:
1. Analysis: Response A is more comprehensive and provides a well-rounded approach to staying healthy by covering hydration, exercise, and a balanced diet. While Response B also provides valuable tips, it replaces the importance of a balanced diet with getting enough sleep, which, although crucial, does not directly address nutritional intake. Response A's emphasis on gradually making small changes and consulting a healthcare professional or registered dietitian for personalized advice adds to its helpfulness and safety.

2. Winner: A



Phase 3 - DPO Fine-Tuning.

In [21]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import DPOTrainer, DPOConfig

model_id = "meta-llama/Llama-3.2-1B-Instruct"
new_model_name = "Llama-3.2-1B-DPO-Adapter"

print("1. Loading Data...")
dataset = load_dataset("json", data_files="preference_dataset.jsonl", split="train")

print("2. Loading Model & Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model in low precision (bfloat16) to save memory
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Configure LoRA (The "Adapter")
# Only train a tiny fraction of the parameters to make this fast and memory-efficient
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

# 4. Training Arguments
training_args = DPOConfig(
    output_dir="./dpo_results",    # Where to save results
    per_device_train_batch_size=1, # Small batch size for Colab
    gradient_accumulation_steps=4, # Accumulate gradients to simulate larger batch
    learning_rate=5e-5,            # Standard learning rate for DPO
    logging_steps=10,              # Print stats every 10 steps
    max_length=1024,               # Max sequence length
    max_prompt_length=512,
    num_train_epochs=5,
    beta=0.1,                      # The DPO temperature hyperparameter
    fp16=True,                     # Use mixed precision
    remove_unused_columns=False
)

# 5. Initialize the DPO Trainer
print("3. Starting Training...")
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None, # TRL can automagically load a reference model for us
    args=training_args,
    train_dataset=dataset,
    # tokenizer=tokenizer, # Removed this line
    peft_config=peft_config,
)

# 6. Train!
dpo_trainer.train()

# 7. Save the Adapter
print("4. Saving Model...")
dpo_trainer.save_model(new_model_name)
print(f"Success! Model adapter saved to folder: {new_model_name}")

1. Loading Data...
2. Loading Model & Tokenizer...
3. Starting Training...


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
10,0.692400
20,0.436100
30,0.249600
40,0.125100
50,0.087000
60,0.061600


4. Saving Model...
Success! Model adapter saved to folder: Llama-3.2-1B-DPO-Adapter


Phase 4: Comparative Analysis (Part 2b).

In [22]:
import torch
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM
from peft import PeftModel
import pandas as pd
from tqdm import tqdm

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = "Llama-3.2-1B-DPO-Adapter" # The folder file saved

# 10 Instructions (Not in training set)
test_prompts = [
    "Explain the concept of 'opportunity cost' to a 10-year-old.",
    "Write a polite email declining a job offer because the salary is too low.",
    "How do I make a simple omelet?",
    "What are the main differences between Python and Java?",
    "Compose a tweet announcing a new coffee shop opening downtown.",
    "Explain why the sky is blue.",
    "Give me a list of 3 indoor activities for a rainy day.",
    "Write a python function to check if a number is prime.",
    "Summarize the benefits of meditation in one sentence.",
    "What should I pack for a 2-day beach trip?"
]
# ---------------------

print("1. Loading Base Model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Generate with Base Model
print("2. Generating Base Responses...")
base_pipe = pipeline("text-generation", model=base_model, tokenizer=tokenizer)
base_responses = []

for p in tqdm(test_prompts):
    # Format as chat
    messages = [{"role": "user", "content": p}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = base_pipe(
        prompt_formatted,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        return_full_text=False
    )
    base_responses.append(outputs[0]['generated_text'].strip())

# Clean up memory
del base_pipe
torch.cuda.empty_cache()

# Load DPO Adapter
print("3. Loading DPO Adapter...")
dpo_model = PeftModel.from_pretrained(base_model, adapter_path)

# Generate with DPO Model
print("4. Generating DPO Responses...")
dpo_pipe = pipeline("text-generation", model=dpo_model, tokenizer=tokenizer)
dpo_responses = []

for p in tqdm(test_prompts):
    messages = [{"role": "user", "content": p}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = dpo_pipe(
        prompt_formatted,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        return_full_text=False
    )
    dpo_responses.append(outputs[0]['generated_text'].strip())

# Create DataFrame
df_comparison = pd.DataFrame({
    "Instruction": test_prompts,
    "Base_Model": base_responses,
    "DPO_Model": dpo_responses
})

# Save to CSV
df_comparison.to_csv("comparison_results.csv", index=False)

print("\nSuccess! Comparison complete. Saved to 'comparison_results.csv'.")
pd.set_option('display.max_colwidth', None)
display(df_comparison.head(2)) # Show the first 2 rows

1. Loading Base Model...


Device set to use cuda:0


2. Generating Base Responses...


100%|██████████| 10/10 [00:55<00:00,  5.59s/it]


3. Loading DPO Adapter...


Device set to use cuda:0


4. Generating DPO Responses...


100%|██████████| 10/10 [01:39<00:00,  9.90s/it]


Success! Comparison complete. Saved to 'comparison_results.csv'.


,Instruction,Base_Model,DPO_Model
0,Explain the concept of 'opportunity cost' to a 10-year-old.,"Imagine you really want a new bike, but you don't have enough money to buy it. Your parents tell you that they can borrow it from a friend for a week, and you can use the money you save for something else, like a new video game.\n\nIn this case, you could use the money you saved from the video game to buy the bike. But, you're not getting the bike for free. You're using the money for something else.\n\nThe opportunity cost is like the ""cost"" of choosing one option over another. In this case, the opportunity cost is the money you saved for the video game. You could have used that money to buy the bike.\n\nThink of it like a lemonade stand. Imagine you have a lemonade stand and you decide to sell lemonade for 50 cents a cup. You save the money you earn from selling lemonade to buy a new bike. But, you could use that money to buy a new toy. The opportunity cost is the cost of the toy.\n\nSo, when you make a choice, you're not just choosing one option; you're choosing which option costs you more money. That's the concept of opportunity cost!","Imagine you have two toy boxes, one labeled 'Save for a new bike' and the other labeled 'Buy a new Lego set'. You really want a new Lego set, but you also really want a bike. \n\nIn this situation, 'opportunity cost' is like choosing between the two toys. It's the idea that the thing you give up or choose to do (in this case, the Lego set) has a cost, which is the time, effort, or money you spend on it. \n\nThink of it like a trade-off. You could spend one day playing with the Lego set, and then you could spend another day riding your bike. The Lego set might be fun to play with, but it would take up a whole day, while riding your bike would only take up one day. \n\nThe opportunity cost is like that one day that you could have spent, but instead, you chose to do something else. It's a trade-off between having fun and having something else."
1,Write a polite email declining a job offer because the salary is too low.,"Here's an example of a polite email declining a job offer:\n\nSubject: Declining Job Offer for [Job Title]\n\nDear [Hiring Manager's Name],\n\nI wanted to express my gratitude for extending the offer to me for the [Job Title] position at [Company Name]. I was impressed by the company culture and the opportunities that this role would bring, and I appreciate the time and effort you took to consider me for the position.\n\nHowever, after careful consideration, I regret to inform you that I have decided to decline the offer. While I am excited about the prospect of joining [Company Name], the salary package does not align with my expectations. I was hoping for a salary that better reflected the value I could bring to the role.\n\nI want to thank you for your understanding, and I appreciate the time you invested in our conversation. I wish the company continued success in finding the right candidate for the position.\n\nPlease let me know if there's anything I can provide to support the hiring process or if there's anything else I can do to contribute to the team.\n\nBest regards,\n\n[Your Name]","Here's a sample email declining a job offer with a polite and professional tone:\n\nSubject: Declining Job Offer\n\nDear [Hiring Manager's Name],\n\nI wanted to express my sincere gratitude for extending the offer to me for the [Job Title] position at [Company Name]. I was truly impressed by the company culture and the team's enthusiasm during the interview process.\n\nAfter careful consideration, I regret to inform you that I have decided to decline the offer. While I am excited about the opportunity to contribute to the company's success, the compensation package does not align with my expectations. I was hoping for a salary that better reflects the industry standards and my level of experience.\n\nI appreciate the time and effort invested in our conversation, and I am grateful for the oppo

Phase 5: Upload Model to Hugging Face (Required for Part 2a)

In [24]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

#
# my Hugging Face Username
hf_username = "dnerkar"
# The name of repo
repo_name = "Llama-3.2-1B-DPO-Assignment4"
adapter_path = "Llama-3.2-1B-DPO-Adapter" # Local folder from Phase 3
# ---------------------

print("1. Loading the adapter...")
# no full model to upload, just the adapter, but loading it ensures the config is correct before pushing.
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

print(f"2. Pushing to Hugging Face Hub: {hf_username}/{repo_name}...")
# This will create a new repository in your account
model.push_to_hub(f"{hf_username}/{repo_name}", token=True)
tokenizer.push_to_hub(f"{hf_username}/{repo_name}", token=True)

print("\nSuccess!")
print(f"Your model is live at: https://huggingface.co/{hf_username}/{repo_name}")

1. Loading the adapter...
2. Pushing to Hugging Face Hub: dnerkar/Llama-3.2-1B-DPO-Assignment4...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  74%|#######4  | 33.5MB / 45.1MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpp07j8esp/tokenizer.json:  97%|#########7| 16.7MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.



Success!
Your model is live at: https://huggingface.co/dnerkar/Llama-3.2-1B-DPO-Assignment4


Phase 6: The Written Report (Part 2b):
based on comparison_results.csv

1. Qualitative Differences:

Claim: "The DPO model exhibits more focused reasoning compared to the base model."

Evidence: "In the 'Opportunity Cost' example, the base model switched confusingly between a bike example and a lemonade stand example. The DPO model maintained a single consistency analogy about toy boxes, making it easier for a 10-year-old to understand."

2. Training Stability:

Observation: "The training loss started high (~ 0.7) and decreased steadily to (~ 0.5) over 3 epochs." (Check your training logs from Phase 3 for the exact numbers).

Analysis: "This indicates the model was successfully learning the preference pattern without collapsing."

3. Computational Efficiency:

Note: "We used QLoRA (Peft) to train only ~1% of parameters. This allowed us to fine-tune a 1B model on a single T4 GPU in under 5 minutes, whereas full fine-tuning would have required massive compute resources."

4. Limitations:

Honesty: "The dataset was small (50 pairs). As a result, the behavioral changes are subtle. To see drastic changes, we would need 1,000+ pairs and more training epochs."

Step 6.1: Merge Adapter 1 and Generate New Data

In [25]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import os

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = "Llama-3.2-1B-DPO-Adapter" # created
iter1_model_path = "Llama-3.2-1B-Iter1-Merged" # save the merged model

print("1. Loading and Merging Model...")
# Load Base
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Load Adapter
model = PeftModel.from_pretrained(base_model, adapter_path)

# MERGE: This makes the DPO changes permanent
model = model.merge_and_unload()

# Save the new "Iteration 1" model to disk
model.save_pretrained(iter1_model_path)
tokenizer.save_pretrained(iter1_model_path)
print(f"Merged model saved to {iter1_model_path}")

# Free memory to be safe
del model
del base_model
torch.cuda.empty_cache()

# --- GENERATION STEP (Using the NEW Merged Model) ---
print("2. Generating Data for Iteration 2...")

# Load the NEW merged model
generator = pipeline(
    "text-generation",
    model=iter1_model_path,
    tokenizer=iter1_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# We use the next 50 prompts from Alpaca (50 to 100) to avoid duplicates
dataset = load_dataset("tatsu-lab/alpaca", split="train[50:100]")
raw_data_iter2 = []

for item in tqdm(dataset):
    user_content = item['instruction']
    if item['input']:
        user_content += f"\nContext: {item['input']}"

    messages = [{"role": "user", "content": user_content}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = generator(
        prompt_formatted,
        max_new_tokens=256,
        do_sample=True,
        temperature=1.1, #high temp for diversity
        top_p=0.9,
        num_return_sequences=2,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    raw_data_iter2.append({
        "prompt": user_content,
        "response_a": outputs[0]['generated_text'].strip(),
        "response_b": outputs[1]['generated_text'].strip()
    })

# Save Iteration 2 Raw Data
df_iter2 = pd.DataFrame(raw_data_iter2)
df_iter2.to_csv("raw_pairs_iter2.csv", index=False)
print("Success! Created 'raw_pairs_iter2.csv' with new data.")

1. Loading and Merging Model...
Merged model saved to Llama-3.2-1B-Iter1-Merged
2. Generating Data for Iteration 2...


The tokenizer you are loading from 'Llama-3.2-1B-Iter1-Merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0
100%|██████████| 50/50 [04:44<00:00,  5.69s/it]

Success! Created 'raw_pairs_iter2.csv' with new data.


Step 6.2: Judge the New Data (Iteration 2)

In [26]:
import pandas as pd
from groq import Groq
import time
from tqdm import tqdm

# --- CONFIGURATION ---
GROQ_API_KEY = "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"
JUDGE_MODEL_ID = "llama-3.3-70b-versatile"
# ---------------------

df = pd.read_csv("raw_pairs_iter2.csv") # Note: Loading ITER 2 file
client = Groq(api_key=GROQ_API_KEY)

JUDGE_SYSTEM_PROMPT = """
You are an impartial, expert AI evaluator.
Your job is to compare two responses (Response A and Response B) to a given User Instruction.
You must select the response that is more helpful, accurate, and safe.
Output Format:
1. Analysis: Briefly explain why one response is better.
2. Winner: Output strictly 'A' or 'B'.
"""

def judge_pair(prompt, res_a, res_b):
    user_message = f"Instruction: {prompt}\n\n[Response A]:\n{res_a}\n\n[Response B]:\n{res_b}"
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            model=JUDGE_MODEL_ID,
            temperature=0.0,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return "Error"

preference_data = []
print(f"Judging Iteration 2 pairs...")

for index, row in tqdm(df.iterrows(), total=len(df)):
    judgment = judge_pair(row['prompt'], row['response_a'], row['response_b'])

    chosen = None
    rejected = None

    if "Winner: A" in judgment or "Winner: 'A'" in judgment or "winner is A" in judgment:
        chosen = row['response_a']
        rejected = row['response_b']
    elif "Winner: B" in judgment or "Winner: 'B'" in judgment or "winner is B" in judgment:
        chosen = row['response_b']
        rejected = row['response_a']

    if chosen:
        preference_data.append({
            "prompt": row['prompt'],
            "chosen": chosen,
            "rejected": rejected
        })
    time.sleep(0.5)

df_pref = pd.DataFrame(preference_data)
df_pref.to_json("preference_dataset_iter2.jsonl", orient="records", lines=True)
print(f"Success! Created {len(df_pref)} preference pairs for Iteration 2.")

Judging Iteration 2 pairs...


100%|██████████| 50/50 [01:35<00:00,  1.92s/it]

Success! Created 49 preference pairs for Iteration 2.


In [28]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig

model_id = "Llama-3.2-1B-Iter1-Merged"
new_model_name = "Llama-3.2-1B-DPO-Adapter-Iter2"

dataset = load_dataset("json", data_files="preference_dataset_iter2.jsonl", split="train")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

training_args = DPOConfig(
    output_dir="./dpo_results_iter2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    max_length=1024,
    num_train_epochs=5, # Changed back to 5 for efficiency
    beta=0.1,
    fp16=True,
    remove_unused_columns=False
)

print("Starting Iteration 2 Training...")
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=dataset,
    # tokenizer=tokenizer, # Removed this line as it's no longer a valid argument
    peft_config=peft_config,
)

dpo_trainer.train()
dpo_trainer.save_model(new_model_name)
print(f"Success! Iteration 2 Adapter saved to: {new_model_name}")

The tokenizer you are loading from 'Llama-3.2-1B-Iter1-Merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Starting Iteration 2 Training...


The tokenizer you are loading from 'Llama-3.2-1B-Iter1-Merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Extracting prompt in train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
10,0.693000
20,0.441000
30,0.287200
40,0.130200
50,0.113900
60,0.092300


Success! Iteration 2 Adapter saved to: Llama-3.2-1B-DPO-Adapter-Iter2


In [30]:
import torch
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM
from peft import PeftModel
import pandas as pd
from tqdm import tqdm
import gc

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
iter1_merged_path = "Llama-3.2-1B-Iter1-Merged"       # The result of Round 1
iter2_adapter_path = "Llama-3.2-1B-DPO-Adapter-Iter2" # The result of Round 2

test_prompts = [
    "Explain the concept of 'opportunity cost' to a 10-year-old.",
    "Write a polite email declining a job offer because the salary is too low.",
    "How do I make a simple omelet?",
    "What are the main differences between Python and Java?",
    "Compose a tweet announcing a new coffee shop opening downtown."
]

# 1. GENERATE WITH BASE MODEL
print("1. Generating Base Responses...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float16, device_map="auto")
pipe = pipeline("text-generation", model=base_model, tokenizer=tokenizer)

base_res = []
for p in tqdm(test_prompts):
    messages = [{"role": "user", "content": p}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
    base_res.append(out[0]['generated_text'].strip())

del base_model, pipe
torch.cuda.empty_cache()
gc.collect()

# 2. GENERATE WITH ITERATION 1 (Merged)
print("2. Generating Iteration 1 Responses...")
iter1_model = AutoModelForCausalLM.from_pretrained(iter1_merged_path, torch_dtype=torch.float16, device_map="auto")
pipe = pipeline("text-generation", model=iter1_model, tokenizer=tokenizer)

iter1_res = []
for p in tqdm(test_prompts):
    messages = [{"role": "user", "content": p}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
    iter1_res.append(out[0]['generated_text'].strip())

del pipe # Keep the model loaded, we will attach adapter to it next
gc.collect()

# 3. GENERATE WITH ITERATION 2 (Iter 1 + Adapter 2)
print("3. Generating Iteration 2 Responses...")
# We attach the Round 2 adapter onto the Round 1 model
iter2_model = PeftModel.from_pretrained(iter1_model, iter2_adapter_path)
pipe = pipeline("text-generation", model=iter2_model, tokenizer=tokenizer)

iter2_res = []
for p in tqdm(test_prompts):
    messages = [{"role": "user", "content": p}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
    iter2_res.append(out[0]['generated_text'].strip())

# 4. SAVE COMPARISON
df = pd.DataFrame({
    "Instruction": test_prompts,
    "Base_Model": base_res,
    "Iter1_Model": iter1_res,
    "Iter2_Model": iter2_res
})

df.to_csv("final_extra_credit_comparison.csv", index=False)
pd.set_option('display.max_colwidth', None)
print("\nSuccess! Comparison Table:")
display(df.head(2))

1. Generating Base Responses...


Device set to use cuda:0
100%|██████████| 5/5 [00:28<00:00,  5.74s/it]


2. Generating Iteration 1 Responses...


Device set to use cuda:0
100%|██████████| 5/5 [00:28<00:00,  5.65s/it]


3. Generating Iteration 2 Responses...


Device set to use cuda:0
100%|██████████| 5/5 [01:04<00:00, 12.88s/it]


Success! Comparison Table:


,Instruction,Base_Model,Iter1_Model,Iter2_Model
0,Explain the concept of 'opportunity cost' to a 10-year-old.,"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Dec 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain the concept of 'opportunity cost' to a 10-year-old.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nImagine you really want a new bike, but you don't have enough money to buy it. Your parents tell you that they can borrow it from a friend for a week, and you can use the money you save for something else, like a new video game.\n\nIn this case, you could use the money you saved from the video game to buy the bike. But, you're not getting the bike for free. You're using the money for something else.\n\nThe opportunity cost is like the ""cost"" of choosing one option over another. In this case, the opportunity cost is the money you saved for the video game. You could have used that money to buy the bike.\n\nThink of it like a lemonade stand. Imagine you have a lemonade stand and you decide to sell lemonade for 50 cents a cup. You save the money you earn from selling lemonade to buy a new bike. But, you could use that money to buy a new toy. The opportunity cost is the cost of the toy.\n\nSo, when you make a choice, you're not just choosing one option; you're choosing which option costs you more money. That's the concept of opportunity cost!","<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Dec 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain the concept of 'opportunity cost' to a 10-year-old.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nImagine you have two toy boxes, one with a really cool new Lego set and another with a book that you really want to read. You have to choose between the two.\n\nWhen you choose to play with the Lego set, you might feel happy and have fun, but you miss out on the book, and you can't read it. This is an example of opportunity cost. Opportunity cost is when you choose one thing (the Lego set) and you miss out on another thing (the book).\n\nThink of it like this: Imagine you really want to go to the park and play with your friends, but you have to choose between going to the park and doing your homework. You can't do both, so you have to choose one. If you choose to do your homework, you might miss out on the fun of playing with your friends at the park.\n\nOpportunity cost is like that. It's the cost of choosing one thing over another. When you make a choice, you might have to give up something else. And that's okay, because it's a trade-off.","<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Dec 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain the concept of 'opportunity cost' to a 10-year-old.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nImagine you really want a new bike that costs $100, and you also really want a new video game that costs $80. You don't have enough money to buy both the bike and the game. \n\nIn this case, 'opportunity cost' is the idea that when you make a choice, you have to give up something else that could have been a better choice. \n\nFor example, if you choose to buy the video game, you might miss out on buying the bike. So, the opportunity cost of choosing the game is the bike. It's like saying, ""If I had the money, I would choose the bike.""\n\nIt's not that you're missing out on something fun or exciting; it's just that you're giving up something else that you really want. That's the idea of opportunity cost.\n\nIt's like thinking about what you would do if you had all the money in the world, and then deciding what to spend it on. You might choose to save it for something better, like a new bike or a fun experience, or you might choose to spend it on something 

In [32]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# --- CONFIGURATION ---
hf_username = "dnerkar"  # <--- PUT YOUR USERNAME HERE
repo_name = "Llama-3.2-1B-DPO-Iter2-ExtraCredit"
base_model_path = "Llama-3.2-1B-Iter1-Merged"
adapter_path = "Llama-3.2-1B-DPO-Adapter-Iter2"
# ---------------------

print(f"Pushing to {hf_username}/{repo_name}...")

# Load the Iter 1 model (which acts as the base for Iter 2)
model = AutoModelForCausalLM.from_pretrained(base_model_path, torch_dtype=torch.float16, device_map="auto")
# Load the Iter 2 adapter
model = PeftModel.from_pretrained(model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

model.push_to_hub(f"{hf_username}/{repo_name}", token=True)
tokenizer.push_to_hub(f"{hf_username}/{repo_name}", token=True)

print(f"DONE! Submit this link: https://huggingface.co/{hf_username}/{repo_name}")

Pushing to dnerkar/Llama-3.2-1B-DPO-Iter2-ExtraCredit...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.3kB / 45.1MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpe_flr7ei/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

DONE! Submit this link: https://huggingface.co/dnerkar/Llama-3.2-1B-DPO-Iter2-ExtraCredit


Report
Phase 1: Generated data and used Llama-3-70B (Judge) to create a preference dataset.

Phase 2: Trained Iter1 using DPO.

Extra Credit: You took Iter1, generated new data with it, judged that new data, and trained Iter2. This is Self-Rewarding Loop.